# 01 - EDA Structural Damage Classification

**Mahasiswa:** Rahardian Putra  
**NRP:** 5052231018

Audit dataset kompetisi dan distribusi kelas.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path("/kaggle/working") if Path("/kaggle/input").exists() else Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
SOURCES = {'scripts/__init__.py': '"""Reproducible structural damage classification workflow."""\n', 'scripts/feature_engineering.py': '"""Deterministic, pre-earthquake feature engineering without target information."""\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import BaseEstimator, TransformerMixin\n\n\nclass BuildingFeatures(TransformerMixin, BaseEstimator):\n    """Remove identifiers and derive structural ratios and categorical geography."""\n\n    def fit(self, X: pd.DataFrame, y=None):\n        """Record the required input schema without learning from validation rows."""\n        self.feature_names_in_ = np.asarray(X.columns, dtype=object)\n        return self\n\n    def transform(self, X: pd.DataFrame) -> pd.DataFrame:\n        """Return independent engineered features while preserving row order."""\n        X = X.loc[:, self.feature_names_in_].copy()\n        X = X.drop(columns=[\'building_id\'], errors=\'ignore\')\n        for column in [\'district_id\', \'vdcmun_id\', \'ward_id\']:\n            X[column] = X[column].astype(\'string\').fillna(\'missing\').astype(object)\n        floors = X[\'count_floors_pre_eq\'].replace(0, np.nan)\n        X[\'height_per_floor\'] = X[\'height_ft_pre_eq\'] / floors\n        X[\'estimated_floor_area\'] = X[\'plinth_area_sq_ft\'] * floors\n        X[\'age_is_999\'] = X[\'age_building\'].eq(999).astype(int)\n        X[\'log_plinth_area\'] = np.log1p(X[\'plinth_area_sq_ft\'].clip(lower=0))\n        return X.replace([np.inf, -np.inf], np.nan)\n', 'scripts/predict.py': '"""Prediction alignment and strict competition submission validation."""\n\nfrom pathlib import Path\nimport pandas as pd\n\nLABELS = [\'Grade 1\', \'Grade 2\', \'Grade 3\', \'Grade 4\', \'Grade 5\']\n\n\ndef create_submission(model, test: pd.DataFrame, sample: pd.DataFrame, output: Path) -> pd.DataFrame:\n    """Predict original string labels and align them to sample IDs before saving."""\n    if list(sample.columns) != [\'building_id\', \'damage_grade\']:\n        raise ValueError(\'Unexpected sample submission columns.\')\n    if len(sample) != len(test) or sample.building_id.isna().any() or test.building_id.isna().any():\n        raise ValueError(\'Invalid test/sample row counts or missing IDs.\')\n    if not test.building_id.is_unique or not sample.building_id.is_unique:\n        raise ValueError(\'Submission IDs must be unique.\')\n    if set(sample.building_id) != set(test.building_id):\n        raise ValueError(\'Sample and test IDs do not match.\')\n    predictions = pd.Series(model.predict(test), index=test.building_id)\n    result = sample.copy()\n    result[\'damage_grade\'] = result.building_id.map(predictions)\n    if result.damage_grade.isna().any() or not result.damage_grade.isin(LABELS).all():\n        raise ValueError(\'Invalid predicted labels.\')\n    output.parent.mkdir(parents=True, exist_ok=True)\n    result.to_csv(output, index=False)\n    reread = pd.read_csv(output)\n    pd.testing.assert_frame_equal(result.reset_index(drop=True), reread)\n    return result\n', 'scripts/preprocess.py': '"""Fold-local preprocessing and non-neural classifiers."""\n\nimport numpy as np\nfrom lightgbm import LGBMClassifier\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler\n\nfrom scripts.feature_engineering import BuildingFeatures\n\n\ndef make_pipeline(X, kind: str, seed: int = 42) -> Pipeline:\n    """Build an unfitted pipeline; the inspected frame determines only column types."""\n    schema = BuildingFeatures().fit_transform(X.head(2))\n    numeric = schema.select_dtypes(include=np.number).columns.tolist()\n    categorical = [c for c in schema if c not in numeric]\n    numeric_steps = [(\'imputer\', SimpleImputer(strategy=\'median\', keep_empty_features=True))]\n    if kind in (\'dummy\', \'logistic\'):\n        numeric_steps.append((\'scale\', StandardScaler()))\n        encoder = OneHotEncoder(handle_unknown=\'ignore\', sparse_output=True)\n    else:\n        encoder = OrdinalEncoder(handle_unknown=\'use_encoded_value\', unknown_value=-1)\n    preprocessor = ColumnTransformer([\n        (\'numeric\', Pipeline(numeric_steps), numeric),\n        (\'categorical\', Pipeline([\n            (\'imputer\', SimpleImputer(strategy=\'most_frequent\', keep_empty_features=True)),\n            (\'encoder\', encoder),\n        ]), categorical),\n    ], remainder=\'drop\', sparse_threshold=1.0)\n    models = {\n        \'dummy\': DummyClassifier(strategy=\'prior\'),\n        \'logistic\': LogisticRegression(max_iter=400, C=1.0, class_weight=\'balanced\', random_state=seed),\n        \'forest\': RandomForestClassifier(n_estimators=100, max_depth=20,\n            min_samples_leaf=3, class_weight=\'balanced_subsample\', n_jobs=2, random_state=seed),\n        \'lightgbm\': LGBMClassifier(n_estimators=350, learning_rate=0.06,\n            num_leaves=63, max_bin=127, colsample_bytree=0.9,\n            reg_lambda=2.0, verbosity=-1, n_jobs=2, random_state=seed,\n            deterministic=True, force_col_wise=True),\n    }\n    pipeline = Pipeline([(\'features\', BuildingFeatures()), (\'preprocess\', preprocessor), (\'model\', models[kind])])\n    # Ordinal codes are categorical, not continuous distances, for LightGBM.\n    pipeline.categorical_indices = list(range(len(numeric), len(numeric) + len(categorical)))\n    return pipeline\n', 'scripts/train.py': '"""EDA, baselines, development-only tuning, held-out evaluation and final refit."""\n\nimport argparse\nimport json\nimport platform\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport joblib\nimport lightgbm\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.base import clone\nfrom sklearn.inspection import permutation_importance\nfrom sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score\nfrom sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split\n\nfrom scripts.preprocess import make_pipeline\nfrom scripts.predict import LABELS, create_submission\n\n\ndef stratified_subset(X, y, limit: int):\n    """Select a reproducible bounded sample without mixing development and holdout."""\n    if len(X) <= limit:\n        return X, y\n    subset, _, labels, _ = train_test_split(X, y, train_size=limit, stratify=y, random_state=42)\n    return subset, labels\n\n\ndef inspect_data(data_dir: Path, output_dir: Path):\n    """Validate the real competition schema and write an EDA audit and figures."""\n    train, test, sample = [pd.read_csv(data_dir / name) for name in\n        [\'train.csv\', \'test.csv\', \'sample_submission.csv\']]\n    if set(train.damage_grade.unique()) != set(LABELS):\n        raise ValueError(\'Unexpected target labels.\')\n    if set(train.columns) - {\'damage_grade\'} != set(test.columns):\n        raise ValueError(\'Train and test feature schemas differ.\')\n    if not train.building_id.is_unique or not test.building_id.is_unique:\n        raise ValueError(\'Repeated building IDs require grouped validation.\')\n    if set(train.building_id) & set(test.building_id):\n        raise ValueError(\'Train and test building IDs overlap.\')\n    output_dir.mkdir(parents=True, exist_ok=True)\n    figures = output_dir / \'reports/figures\'\n    figures.mkdir(parents=True, exist_ok=True)\n    audit = {\n        \'train_shape\': list(train.shape), \'test_shape\': list(test.shape),\n        \'class_counts\': train.damage_grade.value_counts().sort_index().to_dict(),\n        \'missing_train\': train.isna().sum().to_dict(),\n        \'missing_test\': test.isna().sum().to_dict(),\n        \'duplicate_rows_train\': int(train.duplicated().sum()),\n        \'identical_feature_rows_train\': int(train.drop(columns=[\'building_id\', \'damage_grade\']).duplicated().sum()),\n        \'unique_train_ids\': bool(train.building_id.is_unique),\n        \'age_999_count\': int(train.age_building.eq(999).sum()),\n        \'geography_overlap_test\': {c: float(test[c].isin(train[c]).mean()) for c in [\'district_id\',\'vdcmun_id\',\'ward_id\']},\n    }\n    (output_dir / \'reports/data_audit_v1.json\').write_text(json.dumps(audit, indent=2))\n    train.damage_grade.value_counts().sort_index().plot.bar(title=\'Distribusi kelas train\', ylabel=\'Jumlah bangunan\', rot=0)\n    plt.tight_layout()\n    plt.savefig(figures / \'01_distribusi_kelas_v1.png\', dpi=150)\n    plt.close()\n    print(\'DATA_AUDIT\', json.dumps(audit), flush=True)\n    return train, test, sample\n\n\ndef run(data_dir: Path, output_dir: Path, quick: bool = False) -> dict:\n    """Run reproducible experiments without using competition test labels."""\n    train, test, sample = inspect_data(data_dir, output_dir)\n    X, y = train.drop(columns=\'damage_grade\'), train.damage_grade\n    Xdev, Xhold, ydev, yhold = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)\n    pd.DataFrame({\'building_id\': X.building_id,\n        \'split\': np.where(X.index.isin(Xhold.index), \'holdout\', \'development\')}).to_csv(output_dir/\'reports/split.csv\', index=False)\n    Xsmall, ysmall = stratified_subset(Xdev, ydev, 400 if quick else 40000)\n    benchmarks = []\n    for kind in [\'dummy\', \'logistic\', \'forest\']:\n        print(\'BASELINE_START\', kind, flush=True)\n        pipe = make_pipeline(Xsmall, kind)\n        start = perf_counter()\n        pipe.fit(Xsmall, ysmall)\n        score = f1_score(yhold, pipe.predict(Xhold), average=\'macro\')\n        result = {\'model\': kind, \'train_rows\': len(Xsmall), \'holdout_macro_f1\': score,\n                  \'seconds\': perf_counter() - start}\n        benchmarks.append(result)\n        print(\'BASELINE_RESULT\', json.dumps(result), flush=True)\n    base = make_pipeline(Xsmall, \'lightgbm\')\n    categorical_indices = base.categorical_indices\n    params = {\'model__num_leaves\': [31,63], \'model__min_child_samples\': [30,100],\n              \'model__class_weight\': [None, \'balanced\']}\n    if quick:\n        base.set_params(model__n_estimators=10)\n    cv = StratifiedKFold(n_splits=2 if quick else 5, shuffle=True, random_state=42)\n    search = RandomizedSearchCV(base, params, n_iter=1 if quick else 3, scoring=\'f1_macro\',\n        cv=cv, n_jobs=1, random_state=42, refit=True, verbose=1, error_score=\'raise\')\n    print(\'TUNING_START: development subset only; holdout remains outside search\', flush=True)\n    search.fit(Xsmall, ysmall, model__categorical_feature=categorical_indices)\n    pd.DataFrame(search.cv_results_).to_csv(output_dir/\'reports/hyperparameter_search_v1.csv\', index=False)\n    print(\'TUNING_RESULT\', search.best_score_, search.best_params_, flush=True)\n    selected = clone(search.best_estimator_)\n    start = perf_counter()\n    selected.fit(Xdev, ydev, model__categorical_feature=categorical_indices)\n    predicted = selected.predict(Xhold)\n    heldout_score = f1_score(yhold, predicted, average=\'macro\')\n    benchmarks.append({\'model\':\'lightgbm_tuned\', \'train_rows\':len(Xdev),\n                       \'holdout_macro_f1\':heldout_score, \'seconds\':perf_counter()-start})\n    pd.DataFrame(benchmarks).to_csv(output_dir/\'reports/model_comparison_v1.csv\', index=False)\n    report = classification_report(yhold, predicted, labels=LABELS, output_dict=True, zero_division=0)\n    (output_dir/\'reports/classification_report_v1.json\').write_text(json.dumps(report, indent=2))\n    print(\'HOLDOUT_MACRO_F1\', heldout_score, flush=True)\n    print(classification_report(yhold, predicted, labels=LABELS, zero_division=0), flush=True)\n    confusion = confusion_matrix(yhold, predicted, labels=LABELS)\n    ConfusionMatrixDisplay(confusion, display_labels=LABELS).plot(cmap=\'Blues\', xticks_rotation=30)\n    plt.tight_layout()\n    plt.savefig(output_dir/\'reports/figures/05_confusion_matrix_v1.png\', dpi=150)\n    plt.close()\n    Xi, yi = stratified_subset(Xhold, yhold, 100 if quick else 2000)\n    importance = permutation_importance(selected, Xi, yi, scoring=\'f1_macro\', n_repeats=2,\n                                       n_jobs=1, random_state=42)\n    imp = pd.DataFrame({\'feature\':Xi.columns, \'mean\':importance.importances_mean,\n                        \'std\':importance.importances_std}).sort_values(\'mean\', ascending=False)\n    imp.to_csv(output_dir/\'reports/permutation_importance_v1.csv\', index=False)\n    imp.head(12).sort_values(\'mean\').plot.barh(x=\'feature\',y=\'mean\',legend=False,title=\'Permutation importance (holdout Macro F1)\')\n    plt.tight_layout()\n    plt.savefig(output_dir/\'reports/figures/06_permutation_importance_v1.png\', dpi=150)\n    plt.close()\n    print(\'FINAL_REFIT_START\', len(X), flush=True)\n    final = clone(search.best_estimator_)\n    final.fit(X, y, model__categorical_feature=categorical_indices)\n    (output_dir/\'models\').mkdir(exist_ok=True)\n    model_path = output_dir/\'models/pipeline_lightgbm.joblib\'\n    joblib.dump(final, model_path)\n    restored = joblib.load(model_path)\n    assert np.array_equal(final.predict(test.head(10)), restored.predict(test.head(10)))\n    submission = create_submission(restored, test, sample, output_dir/\'submissions/submission.csv\')\n    # Top-level copy is convenient for Kaggle\'s submission selector.\n    submission.to_csv(output_dir/\'submission.csv\', index=False)\n    result = {\'holdout_macro_f1\':heldout_score, \'cv_macro_f1\':search.best_score_,\n        \'cv_std\':float(search.cv_results_[\'std_test_score\'][search.best_index_]),\n        \'best_params\':search.best_params_, \'train_rows\':len(train), \'test_rows\':len(test),\n        \'tuning_rows\':len(Xsmall), \'holdout_rows\':len(Xhold),\n        \'prediction_counts\':submission.damage_grade.value_counts().sort_index().to_dict(),\n        \'public_score\':None, \'private_score\':None,\n        \'versions\':{\'python\':platform.python_version(),\'sklearn\':sklearn.__version__,\n            \'lightgbm\':lightgbm.__version__,\'pandas\':pd.__version__,\'numpy\':np.__version__}}\n    (output_dir/\'reports/results_v1.json\').write_text(json.dumps(result, indent=2))\n    print(\'COMPLETE\', json.dumps(result), flush=True)\n    return result\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--data-dir\', type=Path, default=Path(\'data\'))\n    parser.add_argument(\'--output-dir\', type=Path, default=Path(\'.\'))\n    parser.add_argument(\'--quick\', action=\'store_true\', help=\'Smoke test only; not a competition result.\')\n    args = parser.parse_args()\n    run(args.data_dir, args.output_dir, args.quick)\n'}
for name, source in SOURCES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if Path("/kaggle/input").exists() or not path.exists():
        path.write_text(source)
print("Project ready:", ROOT)


In [ ]:
from scripts.train import inspect_data, run
DATA = Path("/kaggle/input/competitions/structural-damage-classification") if Path("/kaggle/input").exists() else ROOT / "data"
assert (DATA / "train.csv").exists(), f"Dataset belum tersedia: {DATA}"
train, test, sample = inspect_data(DATA, ROOT)
display(train.head())
display(train.damage_grade.value_counts().sort_index())
display(train.isna().sum().to_frame("missing"))
